# IMERG Precipitation Data Analysis

This notebook analyzes precipitation data from IMERG CSV files located in the Data/IMERG folder. The data contains precipitation measurements from multiple stations in the Kathmandu valley for different time periods.

## Objective
- Read and explore precipitation CSV files
- Understand data structure and station locations
- Perform basic statistical analysis
- Visualize precipitation patterns

## 1. Import Required Libraries

In [20]:
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## 2. Set Data Directory Path

In [21]:
# Define the path to the IMERG data directory
# Navigate up from the current notebook location to the Data/IMERG folder
data_dir = "../../Data/IMERG"

# Check if the directory exists
if os.path.exists(data_dir):
    print(f"✅ Data directory found: {os.path.abspath(data_dir)}")
else:
    print(f"❌ Data directory not found: {os.path.abspath(data_dir)}")
    
# Alternative absolute path (uncomment if relative path doesn't work)
# data_dir = r"c:\Users\J01013381\OneDrive - Jackson State University\Research Projects\2025\19 WMS Kathmandu\Data\IMERG"

✅ Data directory found: c:\Users\J01013381\OneDrive - Jackson State University\Research Projects\2025\19 WMS Kathmandu\Data\IMERG


## 3. List Available CSV Files

In [22]:
# List all CSV files in the IMERG data directory
csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
csv_files.sort()

print(f"📊 Found {len(csv_files)} CSV files in the IMERG data directory:")
print("="*50)

for i, file in enumerate(csv_files, 1):
    file_path = os.path.join(data_dir, file)
    file_size = os.path.getsize(file_path) / 1024  # Size in KB
    print(f"{i}. {file} ({file_size:.1f} KB)")
    
print("="*50)

📊 Found 4 CSV files in the IMERG data directory:
1. prec_202308_202308.csv (25.5 KB)
2. prec_202407_202407.csv (25.0 KB)
3. prec_202407_202408.csv (15.7 KB)
4. prec_202409_202409.csv (27.8 KB)


## 4. Filter Columns to Match Flood_Event Data

In [23]:
# Read columns from Flood_Event data to use as filter
flood_event_dir = "../../Results/Floods/Flood_Event"

# Get reference columns from the first matching file
reference_file = os.path.join(flood_event_dir, csv_files[0])
reference_df = pd.read_csv(reference_file)
target_columns = reference_df.columns.tolist()

print(f"🎯 Target columns from Flood_Event data:")
for i, col in enumerate(target_columns, 1):
    print(f"  {i}. {col}")

# Filter and load IMERG data with matching columns only
filtered_data = {}

for file in csv_files:
    period = file.replace('prec_', '').replace('.csv', '')
    
    # Read IMERG data
    imerg_path = os.path.join(data_dir, file)
    imerg_df = pd.read_csv(imerg_path)
    
    # Filter to keep only target columns that exist in IMERG data
    available_cols = [col for col in target_columns if col in imerg_df.columns]
    filtered_df = imerg_df[available_cols].copy()
    filtered_df['Date and Time'] = pd.to_datetime(filtered_df['Date and Time'])
    
    filtered_data[period] = filtered_df
    print(f"✅ {period}: {imerg_df.shape[1]} → {filtered_df.shape[1]} columns")

print(f"\n📊 Filtered data loaded for {len(filtered_data)} periods")

🎯 Target columns from Flood_Event data:
  1. Date and Time
  2. TP001_Nagarjun (Prativa's house)
  3. TP002_Tokha (Ashish Dangol's house
  4. TP003_Lapsephedi
  5. TP005_Kusunti Office
  6. TP006_Bhaktapur (KEC)
  7. TP007_Bhardev
  8. TP008_Okhreni Tipping Bucket
✅ 202308_202308: 10 → 8 columns
✅ 202407_202407: 10 → 8 columns
✅ 202407_202408: 10 → 8 columns
✅ 202409_202409: 10 → 8 columns

📊 Filtered data loaded for 4 periods


## 5. Convert to GSSHA Rainfall Input Format

In [24]:
def create_gssha_rainfall_input(filtered_data, period, target_projection='EPSG:32645', output_dir="../../Results/Floods/Flood_Event"):
    """
    Convert filtered IMERG precipitation data to GSSHA rainfall input format.
    
    Parameters:
    filtered_data (DataFrame): Filtered precipitation data
    period (str): Period identifier (e.g., '202407_202407')
    target_projection (str): Target projection (e.g., 'EPSG:32645')
    output_dir (str): Directory to save the GSSHA input file
    
    Returns:
    str: Path to the created GSSHA input file
    """
    from pyproj import Transformer
    
    if filtered_data.empty:
        print("❌ No data available for GSSHA conversion!")
        return None
    
    print(f"🔧 CONVERTING {period} TO GSSHA RAINFALL INPUT FORMAT")
    print("="*70)
    
    # Get station columns (exclude Date and Time)
    station_cols = [col for col in filtered_data.columns if col != 'Date and Time']
    
    # Load station metadata to get actual coordinates
    metadata_file = "../../Data/Citizen Science/KV-Data/TB_Data_Summary.xlsx"
    
    try:
        station_metadata = pd.read_excel(metadata_file, sheet_name='TP_information')
        print(f"✅ Loaded station metadata from: {metadata_file}")
    except Exception as e:
        print(f"❌ Error loading metadata: {e}")
        return None
    
    # Create transformer from WGS84 to target projection
    coord_transformer = Transformer.from_crs("EPSG:4326", target_projection, always_xy=True)
    
    # Get station coordinates from metadata
    station_coords = {}
    station_descriptions = {}
    
    print(f"📍 Station Coordinates (converted to {target_projection}):")
    print("-" * 60)
    
    for station_col in station_cols:
        # Extract station ID from column name (e.g., 'TP001_Nagarjun...' -> 'TP001')
        station_id = station_col.split('_')[0]
        
        # Find station in metadata
        station_info = station_metadata[station_metadata['Logger_ID'] == station_id]
        
        if not station_info.empty:
            lat = station_info['Latitude'].iloc[0]
            lon = station_info['Longitude'].iloc[0]
            site_info = station_info['Site_Information'].iloc[0] if 'Site_Information' in station_info.columns else station_id
            
            # Convert to UTM coordinates
            easting, northing = coord_transformer.transform(lon, lat)
            
            station_coords[station_col] = (easting, northing)
            station_descriptions[station_col] = f"{station_id} - {site_info}"
            
            print(f"  {station_id:<8} | {easting:>12.2f} {northing:>12.2f} | {site_info}")
        else:
            print(f"  ❌ Warning: No metadata found for {station_id}, using default coordinates")
            # Use default coordinates (center of Kathmandu valley)
            lat, lon = 27.7172, 85.3240
            easting, northing = coord_transformer.transform(lon, lat)
            station_coords[station_col] = (easting, northing)
            station_descriptions[station_col] = f"{station_id} - Default Location"
    
    # Calculate number of gages and time periods
    nrgag = len(station_coords)
    nrpds = len(filtered_data)
    
    print(f"\n📊 GSSHA Parameters:")
    print(f"  Number of rain gages (NRGAG): {nrgag}")
    print(f"  Number of time periods (NRPDS): {nrpds}")
    print(f"  Time range: {filtered_data['Date and Time'].min()} to {filtered_data['Date and Time'].max()}")
    
    # Create GSSHA input file content
    gssha_content = []
    
    # EVENT header
    start_date = filtered_data['Date and Time'].min()
    end_date = filtered_data['Date and Time'].max()
    event_description = f"IMERG Precipitation Data {period} - {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}"
    
    gssha_content.append(f'EVENT "{event_description}"')
    gssha_content.append(f"NRPDS {nrpds}")
    gssha_content.append(f"NRGAG {nrgag}")
    
    # COORD cards for each station
    for station_col, (easting, northing) in station_coords.items():
        description = station_descriptions[station_col]
        gssha_content.append(f'COORD {easting:.2f} {northing:.2f} "{description}"')
    
    # GAGES cards for each time period
    print(f"\n📝 Creating GAGES cards for {nrpds} time periods...")
    
    for idx, row in filtered_data.iterrows():
        # Format date and time for GSSHA (YYYY MM DD HH MM)
        dt = row['Date and Time']
        date_str = f"{dt.year:04d} {dt.month:02d} {dt.day:02d} {dt.hour:02d} {dt.minute:02d}"
        
        # Get precipitation values for all stations at this time
        precip_values = []
        for station_col in station_coords.keys():
            precip_val = row[station_col]
            if pd.isna(precip_val):
                precip_val = 0.0  # Replace NaN with 0
            precip_values.append(f"{precip_val:.2f}")
        
        # Create GAGES line
        precip_str = "  ".join(precip_values)
        gssha_content.append(f"GAGES {date_str}  {precip_str}")
    
    # Write to file
    gssha_filename = f"rainfall_input_imerg_{period}.gag"
    gssha_filepath = os.path.join(output_dir, gssha_filename)
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    with open(gssha_filepath, 'w') as f:
        for line in gssha_content:
            f.write(line + '\n')
    
    print(f"\n💾 GSSHA RAINFALL INPUT FILE CREATED:")
    print(f"  File: {gssha_filepath}")
    print(f"  Size: {len(gssha_content)} lines")
    
    # Display first few lines as preview
    print(f"\n📄 PREVIEW (first 10 lines):")
    print("-" * 70)
    for i, line in enumerate(gssha_content[:10], 1):
        print(f"{i:3d}: {line}")
    
    if len(gssha_content) > 10:
        print(f"     ... ({len(gssha_content) - 10} more lines)")
    
    return gssha_filepath

print("✅ GSSHA conversion function defined")

✅ GSSHA conversion function defined


In [25]:
# Convert all filtered datasets to GSSHA format
print("🚀 CONVERTING ALL FILTERED DATASETS TO GSSHA FORMAT")
print("="*70)

gssha_files = []

for period, data in filtered_data.items():
    print(f"\n📊 Processing period: {period}")
    
    gssha_file = create_gssha_rainfall_input(
        data, 
        period, 
        target_projection='EPSG:32645'
    )
    
    if gssha_file:
        gssha_files.append(gssha_file)
        print(f"✅ Created: {os.path.basename(gssha_file)}")
    else:
        print(f"❌ Failed to create GSSHA file for {period}")

print(f"\n🎉 SUMMARY:")
print(f"Successfully created {len(gssha_files)} GSSHA rainfall input files:")
for file in gssha_files:
    print(f"  📁 {os.path.basename(file)}")

print(f"\n📂 Files saved to: ../../Results/Floods/Flood_Event/")

🚀 CONVERTING ALL FILTERED DATASETS TO GSSHA FORMAT

📊 Processing period: 202308_202308
🔧 CONVERTING 202308_202308 TO GSSHA RAINFALL INPUT FORMAT
✅ Loaded station metadata from: ../../Data/Citizen Science/KV-Data/TB_Data_Summary.xlsx
📍 Station Coordinates (converted to EPSG:32645):
------------------------------------------------------------
  TP001    |    331346.54   3069353.42 | Nagarjun (Prativa's house)
  TP002    |    335315.79   3072831.61 | Tokha (Ashish Dangol's house
  TP003    |    350294.07   3070053.60 | Lapsephedi
  TP005    |    333728.04   3061377.09 | Kusunti Office
  TP006    |    346054.81   3061759.11 | Bhaktapur (KEC)
  TP007    |    340124.57   3048330.45 | Bhardev
  TP008    |    344556.68   3075663.76 | Okhreni Tipping Bucket

📊 GSSHA Parameters:
  Number of rain gages (NRGAG): 7
  Number of time periods (NRPDS): 155
  Time range: 2023-08-06 00:15:00 to 2023-08-09 05:15:00

📝 Creating GAGES cards for 155 time periods...

💾 GSSHA RAINFALL INPUT FILE CREATED:
  Fil